# Notebook 4: Gradient Descent on Cross-Entropy Loss — Full Derivation

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours | **Week 6, Notebook 4**

---

## Why This Matters

In Notebook 3 you learned *what* cross-entropy loss is and *why* it is the right choice for classification. Now comes the crucial engineering question: **how do we actually minimize it?**

The answer is gradient descent — but to implement it, we need the gradient of the loss with respect to every parameter. Deriving this gradient step-by-step reveals one of the most elegant results in machine learning:

$$\frac{\partial \mathcal{L}}{\partial \theta} = \frac{1}{n} X^T(\hat{y} - y)$$

**Plain English:** the gradient is simply the *average prediction error weighted by the input features*. Identical in form to linear regression's gradient — just with a different forward pass.

You will also learn regularization (L1 and L2), which prevents overfitting on high-dimensional data like spam word features, and you will implement the full logistic regression training loop from scratch, verifying it matches sklearn exactly.

---

## Roadmap
1. Full gradient derivation via chain rule (step by step)
2. The update rule and its elegant form
3. Implement `LogisticRegressionGD` from scratch
4. Numerical gradient check (sanity test)
5. Verify against sklearn
6. L2 regularization: shrinkage and the regularization path
7. L1 regularization: sparsity for high-dimensional features
8. Learning rate effects: slow / good / diverges
9. Decision boundary visualization
10. Self-Check (3 questions)

## Real-World Analogy First: The Email Filter Tuning Dial

Imagine your spam filter has **dials** — one for each word feature ("free", "click", "urgent", etc.). Each dial represents a weight $\theta_j$.

After every batch of emails, you check: "How wrong was I, and in which direction?" You then **nudge each dial** a little bit in the direction that would have made the prediction better. The size of the nudge depends on:

1. **How wrong you were** (ŷ - y): large error → big nudge
2. **How strongly that word appeared in the email** (feature $x_j$): words that appeared a lot carry more blame/credit
3. **Your learning rate** $\alpha$: how aggressively to turn the dial

This is exactly what the gradient tells you: $\frac{\partial L}{\partial \theta_j} = \frac{1}{n} \sum_i (\hat{y}_i - y_i) \cdot x_{ij}$

**Regularization** adds a rubber band pulling each dial back toward zero — preventing any single word from dominating and making the filter generalize to new emails.

In [ ]:
# ============================================================
# Cell 1: Setup — Imports, plotting style, and spam dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Consistent color palette
SPAM_COLOR = '#e74c3c'   # red
HAM_COLOR  = '#2ecc71'   # green
C_SLOW     = '#3498db'   # blue  → slow learning rate
C_GOOD     = '#27ae60'   # green → good learning rate
C_FAST     = '#e74c3c'   # red   → diverging learning rate

plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#ffffff',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

# ---------- Rebuild spam dataset (identical to Notebook 3) ----------
np.random.seed(42)
n_spam, n_ham = 250, 250

spam_df = pd.DataFrame({
    'word_count':        np.random.normal(150, 40, n_spam).clip(20, 400),
    'exclamation_marks': np.random.poisson(8,  n_spam),
    'capital_ratio':     np.random.beta(5, 2,  n_spam),
    'link_count':        np.random.poisson(5,  n_spam),
    'label': 1
})
ham_df = pd.DataFrame({
    'word_count':        np.random.normal(350, 80, n_ham).clip(50, 800),
    'exclamation_marks': np.random.poisson(1,  n_ham),
    'capital_ratio':     np.random.beta(2, 5,  n_ham),
    'link_count':        np.random.poisson(1,  n_ham),
    'label': 0
})

df   = pd.concat([spam_df, ham_df]).sample(frac=1, random_state=42).reset_index(drop=True)
X_raw = df[['word_count', 'exclamation_marks', 'capital_ratio', 'link_count']].values
y     = df['label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset ready: {len(X_train)} train, {len(X_test)} test")
print(f"Features: {list(df.columns[:-1])}")

## Part 1: Full Gradient Derivation — Step by Step

We want to find $\frac{\partial \mathcal{L}}{\partial \theta}$ where:

$$\mathcal{L}(\theta) = -\frac{1}{n} \sum_{i=1}^{n} \left[y_i \log \hat{y}_i + (1-y_i) \log(1-\hat{y}_i)\right]$$

We use the **chain rule** across three composed functions:

$$\frac{\partial \mathcal{L}}{\partial \theta} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial \theta}$$

---

### Step 1: $\frac{\partial \mathcal{L}}{\partial \hat{y}}$ — Derivative of Loss w.r.t. Predictions

$$\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = -\frac{1}{n}\left(\frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i}\right) = -\frac{1}{n} \cdot \frac{y_i - \hat{y}_i}{\hat{y}_i(1-\hat{y}_i)}$$

### Step 2: $\frac{\partial \hat{y}}{\partial z}$ — Derivative of Sigmoid w.r.t. Linear Combination

$$\hat{y}_i = \sigma(z_i) = \frac{1}{1+e^{-z_i}}$$

The sigmoid's derivative has a beautiful self-referential form:
$$\frac{\partial \hat{y}_i}{\partial z_i} = \sigma(z_i)(1-\sigma(z_i)) = \hat{y}_i(1-\hat{y}_i)$$

### Step 3: $\frac{\partial z}{\partial \theta}$ — Derivative of Linear Combination

$$z_i = \theta^T x_i \implies \frac{\partial z_i}{\partial \theta} = x_i$$

### The Beautiful Cancellation

Putting it together for a single sample $i$:

$$\frac{\partial \mathcal{L}}{\partial \theta} = \underbrace{-\frac{1}{n} \cdot \frac{y_i - \hat{y}_i}{\hat{y}_i(1-\hat{y}_i)}}_{\partial \mathcal{L}/\partial \hat{y}} \cdot \underbrace{\hat{y}_i(1-\hat{y}_i)}_{\partial \hat{y}/\partial z} \cdot \underbrace{x_i}_{\partial z/\partial \theta}$$

The $\hat{y}_i(1-\hat{y}_i)$ terms **cancel perfectly**, leaving:

$$\boxed{\frac{\partial \mathcal{L}}{\partial \theta} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot x_i = \frac{1}{n} X^T(\hat{y} - y)}$$

**This is remarkable:** The gradient is just the average prediction error times the input features. The sigmoid's complexity completely disappears. And it is identical in form to the linear regression gradient — the model's architecture changed, but the optimization structure is the same.

## Part 2: The Update Rule

### Standard Gradient Descent

$$\theta \leftarrow \theta - \alpha \cdot \frac{1}{n} X^T(\sigma(X\theta) - y)$$

where $\alpha$ is the **learning rate** (step size).

### With L2 Regularization

Add $\frac{\lambda}{n} \|\theta\|^2$ to the loss (penalize large weights). The gradient gains a term:

$$\theta \leftarrow \theta - \alpha \cdot \left[\frac{1}{n} X^T(\hat{y} - y) + \lambda \theta\right]$$

**Note:** The bias term $\theta_0$ is conventionally **not regularized** (set its regularization to 0).

**sklearn convention:** sklearn uses `C = 1/λ` (inverse regularization strength). Large C = small λ = **less** regularization.

### With L1 Regularization

Instead of $\lambda\|\theta\|^2$, use $\lambda\|\theta\|_1 = \lambda \sum_j |\theta_j|$.

L1 gradient = $\lambda \cdot \text{sign}(\theta_j)$ — pushes weights exactly to zero (sparsity!).

This is ideal for spam detection with 10,000+ word features: L1 selects only the handful of words that truly predict spam, zeroing out the rest.

In [ ]:
# ============================================================
# Cell 2: LogisticRegressionGD — Full from-scratch implementation
# Includes: bias term, L2 regularization, loss history, predict_proba
# ============================================================

class LogisticRegressionGD:
    """
    Logistic Regression trained via gradient descent on cross-entropy loss.
    
    Parameters
    ----------
    lr      : float  — learning rate (step size for gradient descent)
    n_iter  : int    — number of gradient descent iterations
    lambda_ : float  — L2 regularization strength (0 = no regularization)
    """
    def __init__(self, lr=0.1, n_iter=1000, lambda_=0.0):
        self.lr      = lr
        self.n_iter  = n_iter
        self.lambda_ = lambda_

    def _sigmoid(self, z):
        """Sigmoid with clipping to prevent overflow in exp(-z) for large z."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        """
        Train the model using gradient descent.
        Gradient formula: (1/n) * Xᵀ(ŷ - y)  [with bias in the design matrix]
        """
        # Prepend a column of ones to X so theta[0] acts as the bias term
        X_b = np.column_stack([np.ones(len(X)), X])  # shape: (n, d+1)
        n   = len(y)

        # Initialize all weights to zero (unbiased starting point)
        self.theta_ = np.zeros(X_b.shape[1])          # shape: (d+1,)
        self.loss_history_ = []

        for _ in range(self.n_iter):
            # --- Forward pass ---
            z     = X_b @ self.theta_                 # linear combination: (n,)
            y_hat = self._sigmoid(z)                   # predicted probabilities: (n,)

            # --- Gradient of cross-entropy: (1/n) Xᵀ(ŷ - y) ---
            grad = (1/n) * X_b.T @ (y_hat - y)        # shape: (d+1,)

            # --- L2 regularization gradient: lambda * theta ---
            # Create a copy to avoid modifying theta in-place before the update
            reg       = self.lambda_ * self.theta_.copy()
            reg[0]    = 0.0   # bias term is NOT regularized (standard practice)

            # --- Parameter update ---
            self.theta_ -= self.lr * (grad + reg)

            # --- Compute and record loss (clip for numerical stability) ---
            y_safe = np.clip(y_hat, 1e-15, 1 - 1e-15)
            loss   = -np.mean(y * np.log(y_safe) + (1 - y) * np.log(1 - y_safe))
            # Add L2 penalty to loss (for monitoring purposes)
            loss  += (self.lambda_ / 2) * np.sum(self.theta_[1:]**2)  # skip bias
            self.loss_history_.append(loss)

        return self  # enables method chaining: model.fit(X, y).predict(X)

    def predict_proba(self, X):
        """Return predicted probability P(spam) for each sample."""
        X_b = np.column_stack([np.ones(len(X)), X])
        return self._sigmoid(X_b @ self.theta_)

    def predict(self, X, threshold=0.5):
        """Return binary predictions: 1 if P(spam) >= threshold, else 0."""
        return (self.predict_proba(X) >= threshold).astype(int)

    @property
    def coef_(self):
        """Feature weights (excluding bias), matching sklearn's API."""
        return self.theta_[1:]

    @property
    def intercept_(self):
        """Bias term, matching sklearn's API."""
        return self.theta_[0]

print("LogisticRegressionGD class defined.")
print("Key design choices:")
print("  - Prepend ones column → bias included in matrix ops")
print("  - gradient = (1/n) Xᵀ(ŷ - y)  [from chain rule derivation]")
print("  - L2 regularization on feature weights only (not bias)")
print("  - Sigmoid clipping prevents exp() overflow")

In [ ]:
# ============================================================
# Cell 3: Train on spam dataset and plot loss curve
# ============================================================

# Train the model (no regularization first, for clean comparison with sklearn)
model_scratch = LogisticRegressionGD(lr=0.1, n_iter=1000, lambda_=0.0)
model_scratch.fit(X_train, y_train)

# Evaluate performance
y_pred_scratch = model_scratch.predict(X_test)
y_prob_scratch = model_scratch.predict_proba(X_test)
acc_scratch    = accuracy_score(y_test, y_pred_scratch)
ce_scratch     = log_loss(y_test, y_prob_scratch)

print("From-Scratch Model Results:")
print(f"  Test Accuracy: {acc_scratch*100:.1f}%")
print(f"  Test Log-Loss: {ce_scratch:.4f}")
print(f"  Learned theta (bias + 4 weights): {model_scratch.theta_.round(4)}")
print(f"  Bias term: {model_scratch.intercept_:.4f}")
print(f"  Feature weights: {model_scratch.coef_.round(4)}")

# Plot loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LogisticRegressionGD: Training on Spam Dataset',
             fontsize=13, fontweight='bold')

# Full loss curve
ax = axes[0]
ax.plot(model_scratch.loss_history_, color=SPAM_COLOR, lw=2)
ax.set_xlabel('Gradient Descent Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title(f'Loss Convergence (lr={model_scratch.lr}, n_iter={model_scratch.n_iter})')
ax.axhline(model_scratch.loss_history_[-1], color='gray', lw=1, linestyle='--',
           label=f'Final loss: {model_scratch.loss_history_[-1]:.4f}')
ax.legend()

# Early iterations (zoom in on initial rapid descent)
ax2 = axes[1]
ax2.plot(model_scratch.loss_history_[:200], color=SPAM_COLOR, lw=2)
ax2.set_xlabel('Iteration (first 200)')
ax2.set_ylabel('Cross-Entropy Loss')
ax2.set_title('Early Iterations — Rapid Initial Descent')
ax2.annotate('Steepest descent\nin first 50 steps',
             xy=(30, model_scratch.loss_history_[30]),
             xytext=(80, model_scratch.loss_history_[5]*0.9), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='black'))

plt.tight_layout()
plt.show()

## Part 3: Numerical Gradient Check

Before trusting any gradient implementation, always **verify it numerically**. The idea is simple:

For a scalar function $f(\theta)$, the gradient at $\theta_j$ can be approximated by the **finite difference**:

$$\frac{\partial f}{\partial \theta_j} \approx \frac{f(\theta + \epsilon \mathbf{e}_j) - f(\theta - \epsilon \mathbf{e}_j)}{2\epsilon}$$

where $\mathbf{e}_j$ is the unit vector in direction $j$ and $\epsilon$ is a tiny perturbation (e.g., $10^{-5}$).

If our analytical gradient (from the chain rule) matches this numerical approximation to within $10^{-5}$, the derivation is correct.

This is called a **gradient check** and is standard practice when implementing custom gradient computations.

In [ ]:
# ============================================================
# Cell 4: Numerical Gradient Check
# Verify: analytical gradient ≈ finite-difference gradient
# If max difference < 1e-5, our chain rule derivation is correct
# ============================================================

def bce_loss_with_theta(theta, X_b, y, lambda_=0.0):
    """
    Compute BCE loss for given theta on dataset (X_b, y).
    X_b already includes the bias column (ones prepended).
    """
    z     = X_b @ theta
    y_hat = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    y_s   = np.clip(y_hat, 1e-15, 1 - 1e-15)
    ce    = -np.mean(y * np.log(y_s) + (1 - y) * np.log(1 - y_s))
    reg   = (lambda_ / 2) * np.sum(theta[1:]**2)  # L2 penalty (skip bias)
    return ce + reg

def analytical_gradient(theta, X_b, y, lambda_=0.0):
    """
    Analytical gradient from chain rule: (1/n) Xᵀ(ŷ - y) + lambda*theta
    (bias term not regularized)
    """
    n     = len(y)
    z     = X_b @ theta
    y_hat = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    grad  = (1/n) * X_b.T @ (y_hat - y)    # core gradient formula
    reg   = lambda_ * theta.copy()
    reg[0] = 0.0   # do not regularize bias
    return grad + reg

def numerical_gradient(theta, X_b, y, lambda_=0.0, eps=1e-5):
    """
    Finite-difference gradient approximation.
    For each dimension j: [f(theta + eps*ej) - f(theta - eps*ej)] / (2*eps)
    """
    num_grad = np.zeros_like(theta)
    for j in range(len(theta)):
        theta_plus  = theta.copy(); theta_plus[j]  += eps
        theta_minus = theta.copy(); theta_minus[j] -= eps
        f_plus  = bce_loss_with_theta(theta_plus,  X_b, y, lambda_)
        f_minus = bce_loss_with_theta(theta_minus, X_b, y, lambda_)
        num_grad[j] = (f_plus - f_minus) / (2 * eps)
    return num_grad

# Run gradient check at the model's trained parameters
X_b_train  = np.column_stack([np.ones(len(X_train)), X_train])
theta_test = model_scratch.theta_.copy()  # use trained weights as test point

analytical = analytical_gradient(theta_test, X_b_train, y_train)
numerical  = numerical_gradient( theta_test, X_b_train, y_train)

max_diff   = np.max(np.abs(analytical - numerical))
rel_error  = np.max(np.abs(analytical - numerical) / (np.abs(analytical) + 1e-8))

print("Gradient Check Results")
print("=" * 50)
print(f"{'Parameter':<15} {'Analytical':>15} {'Numerical':>15} {'Diff':>12}")
print("-" * 57)
param_names = ['bias'] + [f'theta_{i}' for i in range(len(analytical)-1)]
for name, a, n in zip(param_names, analytical, numerical):
    diff = abs(a - n)
    ok   = 'OK' if diff < 1e-5 else 'WARNING!'
    print(f"{name:<15} {a:>15.8f} {n:>15.8f} {diff:>10.2e}  {ok}")

print(f"\nMax absolute difference: {max_diff:.2e}")
print(f"Max relative error:      {rel_error:.2e}")
print(f"\nGradient check {'PASSED' if max_diff < 1e-4 else 'FAILED'}: "
      f"max diff = {max_diff:.2e} {'< 1e-4 ✓' if max_diff < 1e-4 else '> 1e-4 — check your code!'}")

## Part 4: Verify Against sklearn

A further sanity check: train sklearn's `LogisticRegression` (which uses the same cross-entropy loss, just with a more sophisticated optimizer) and compare the learned coefficients.

In [ ]:
# ============================================================
# Cell 5: Compare from-scratch implementation vs sklearn
# Verify coefficients are close and accuracy matches
# ============================================================

# sklearn: C=1e8 means λ≈0 (no regularization), matching our lambda_=0
clf_sklearn = LogisticRegression(C=1e8, max_iter=5000, solver='lbfgs',
                                  random_state=42, fit_intercept=True)
clf_sklearn.fit(X_train, y_train)

# Collect coefficients for comparison
coef_scratch = model_scratch.coef_           # shape (4,) — feature weights only
coef_sklearn = clf_sklearn.coef_[0]          # shape (4,)
bias_scratch = model_scratch.intercept_
bias_sklearn = clf_sklearn.intercept_[0]

# Performance comparison
acc_sklearn = accuracy_score(y_test, clf_sklearn.predict(X_test))
ce_sklearn  = log_loss(y_test, clf_sklearn.predict_proba(X_test)[:, 1])

print("Coefficient Comparison: From-Scratch vs sklearn")
print("=" * 62)
print(f"{'Feature':<22} {'Ours':>12} {'sklearn':>12} {'Close?':>10}")
print("-" * 58)
feat_names = ['word_count', 'exclamation', 'capital_ratio', 'link_count']
for name, c_s, c_sk in zip(feat_names, coef_scratch, coef_sklearn):
    close = np.isclose(c_s, c_sk, atol=0.05)
    print(f"{name:<22} {c_s:>12.5f} {c_sk:>12.5f} {'  ✓' if close else '  ✗ (diff)'}")
print(f"{'bias (intercept)':<22} {bias_scratch:>12.5f} {bias_sklearn:>12.5f} "
      f"{'  ✓' if np.isclose(bias_scratch, bias_sklearn, atol=0.05) else '  ✗'}")

print(f"\nnp.allclose(coefs, atol=0.05): {np.allclose(coef_scratch, coef_sklearn, atol=0.05)}")

print(f"\nPerformance Comparison:")
print(f"  From-scratch — Accuracy: {acc_scratch*100:.1f}%, Log-Loss: {ce_scratch:.4f}")
print(f"  sklearn      — Accuracy: {acc_sklearn*100:.1f}%, Log-Loss: {ce_sklearn:.4f}")
print(f"\nSmall differences are expected: sklearn uses L-BFGS (second-order optimizer)")
print(f"while ours uses plain gradient descent. Both solve the same objective.")

# Visualization: coefficient comparison bar chart
fig, ax = plt.subplots(figsize=(10, 4))
x       = np.arange(len(feat_names))
width   = 0.35
ax.bar(x - width/2, coef_scratch, width, label='From Scratch (GD)', color=SPAM_COLOR, alpha=0.85)
ax.bar(x + width/2, coef_sklearn,  width, label='sklearn (L-BFGS)',  color='steelblue', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(feat_names, rotation=20)
ax.set_ylabel('Coefficient Value')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Learned Coefficients: From-Scratch vs sklearn\n'
             'Very similar — same objective, different optimizers')
ax.legend()
plt.tight_layout()
plt.show()

## Part 5: L2 Regularization — The Regularization Path

**What is regularization?** It adds a penalty to the loss for large weights, preventing the model from fitting noise in the training data.

**L2 (Ridge) regularization:** Adds $\lambda \|\theta\|^2$ to the loss.
- **Small λ** → weights can be large → possible overfitting
- **Large λ** → weights shrink toward zero → underfitting
- **Optimal λ** → sweet spot between fit and generalization

**The regularization path** shows how each coefficient changes as we increase λ from near-zero to very large. All coefficients shrink toward zero, but at different rates depending on how much each feature helps the model.

**sklearn's C parameter:** `C = 1/λ`. So:
- `C = 1e6` → λ ≈ 0 → virtually no regularization
- `C = 1.0` → λ = 1.0 → moderate regularization (sklearn default)
- `C = 0.01` → λ = 100 → strong regularization (heavily shrunk weights)

In [ ]:
# ============================================================
# Cell 6: L2 Regularization Path
# Train with λ = 0, 0.01, 0.1, 1.0, 10.0
# Show coefficient shrinkage and accuracy vs regularization
# ============================================================

lambda_vals = [0.0, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
feat_names  = ['word_count', 'exclamation', 'capital_ratio', 'link_count']
colors_reg  = ['#1a1a2e', '#16213e', '#0f3460', '#533483', '#e94560', '#e76f51', '#f4a261']

coef_path   = {}   # {feature_name: [coef at each lambda]}
acc_path    = []   # test accuracy at each lambda
loss_path   = []   # test CE loss at each lambda

for lam in lambda_vals:
    m = LogisticRegressionGD(lr=0.1, n_iter=1500, lambda_=lam)
    m.fit(X_train, y_train)
    acc_path.append(accuracy_score(y_test, m.predict(X_test)))
    loss_path.append(log_loss(y_test, np.clip(m.predict_proba(X_test), 1e-15, 1-1e-15)))
    for i, name in enumerate(feat_names):
        coef_path.setdefault(name, []).append(m.coef_[i])

print("L2 Regularization Path Results")
print("=" * 60)
print(f"{'λ':>8} {'C=1/λ':>10} {'word_ct':>10} {'exclam':>10} {'cap_rt':>10} {'links':>10} {'Acc':>8}")
print("-" * 60)
for i, lam in enumerate(lambda_vals):
    C_val = f"{1/lam:.2f}" if lam > 0 else "∞"
    coefs = [coef_path[n][i] for n in feat_names]
    print(f"{lam:>8.2f} {C_val:>10} {coefs[0]:>10.4f} {coefs[1]:>10.4f} "
          f"{coefs[2]:>10.4f} {coefs[3]:>10.4f} {acc_path[i]*100:>7.1f}%")

# Regularization path plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('L2 Regularization Path — Spam Classifier\n'
             'As λ increases, all coefficients shrink toward zero',
             fontsize=13, fontweight='bold')

# Panel 1: Coefficient paths
ax = axes[0]
feat_colors = [SPAM_COLOR, '#3498db', '#2ecc71', '#9b59b6']
for i, name in enumerate(feat_names):
    ax.plot(lambda_vals, coef_path[name], marker='o', lw=2,
            color=feat_colors[i], label=name)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('Regularization Strength λ')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Shrinkage with L2 Regularization')
ax.legend(fontsize=9)
ax.set_xscale('symlog', linthresh=0.01)  # log-ish scale

# Panel 2: Accuracy and loss vs lambda
ax2 = axes[1]
ax2.plot(lambda_vals, [a*100 for a in acc_path], marker='s', lw=2,
         color=SPAM_COLOR, label='Test Accuracy (%)')
ax2.set_xlabel('Regularization Strength λ')
ax2.set_ylabel('Test Accuracy (%)', color=SPAM_COLOR)
ax2.tick_params(axis='y', labelcolor=SPAM_COLOR)
ax2.set_title('Test Performance vs Regularization Strength')
ax2.set_xscale('symlog', linthresh=0.01)

ax3 = ax2.twinx()
ax3.plot(lambda_vals, loss_path, marker='^', lw=2, linestyle='--',
         color='steelblue', label='Test Log-Loss')
ax3.set_ylabel('Test Log-Loss', color='steelblue')
ax3.tick_params(axis='y', labelcolor='steelblue')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 7: Bar chart of coefficient magnitudes at different λ values
# Visualizes how L2 shrinks coefficients — easy to read
# ============================================================

selected_lambdas = [0.0, 0.1, 1.0, 10.0]  # four key regularization levels
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
fig.suptitle('Coefficient Magnitudes at Different L2 Regularization Strengths\n'
             'Spam Classifier — 4 Features', fontsize=13, fontweight='bold')

for ax, lam in zip(axes, selected_lambdas):
    idx    = lambda_vals.index(lam)
    coefs  = [coef_path[n][idx] for n in feat_names]
    colors = [SPAM_COLOR if c > 0 else 'steelblue' for c in coefs]
    bars   = ax.bar(feat_names, coefs, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'λ = {lam}\n(C = {"∞" if lam==0 else f"{1/lam:.1f}"})\nAcc: {acc_path[idx]*100:.1f}%')
    ax.set_xticklabels(feat_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Coefficient' if ax == axes[0] else '')
    # Add value labels on bars
    for bar, coef in zip(bars, coefs):
        height = bar.get_height()
        ax.annotate(f'{coef:.3f}',
                   xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords='offset points',
                   ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print("Key observation: as λ increases, all coefficients shrink toward zero.")
print("The model trades off fit quality for smoother, more generalizable weights.")

## Part 6: L1 Regularization — Sparsity for High-Dimensional Features

**L1 (Lasso) regularization** adds $\lambda \|\theta\|_1 = \lambda \sum_j |\theta_j|$ to the loss.

Unlike L2 (which pushes weights toward zero but rarely exactly reaches zero), **L1 drives weights to exactly zero** — producing **sparse solutions**.

### Why L1 Matters for Spam Detection

In a real spam classifier, you might have 10,000+ word features (a bag-of-words representation). Most words are not predictive of spam:
- "the", "and", "is" → not predictive (same frequency in spam and ham)
- "free", "click", "winner" → predictive of spam
- "meeting", "report", "attached" → predictive of ham

**L1 automatically selects** only the truly predictive features, zeroing out irrelevant ones. This is:
- **Interpretable:** only a small set of words drive the decision
- **Efficient:** no need to store weights for thousands of irrelevant features
- **Robust:** less chance of overfitting to noise in feature-rich data

In [ ]:
# ============================================================
# Cell 8: L1 vs L2 Regularization — Sparsity Demonstration
# Use sklearn (L1 gradient descent is tricky; sklearn uses liblinear)
# ============================================================

# sklearn handles L1 internally via coordinate descent / liblinear
# Our manual implementation would need subgradient handling for |θ|

C_vals = [100.0, 10.0, 1.0, 0.5, 0.1, 0.05, 0.01]  # C = 1/λ

coef_l1, coef_l2 = [], []
zero_count_l1 = []

for C in C_vals:
    # L2 (Ridge): smooth shrinkage
    m_l2 = LogisticRegression(C=C, penalty='l2', max_iter=2000, solver='lbfgs')
    m_l2.fit(X_train, y_train)
    coef_l2.append(m_l2.coef_[0].copy())

    # L1 (Lasso): exact zeros
    m_l1 = LogisticRegression(C=C, penalty='l1', max_iter=2000, solver='liblinear')
    m_l1.fit(X_train, y_train)
    coef_l1.append(m_l1.coef_[0].copy())
    zero_count_l1.append(np.sum(np.abs(m_l1.coef_[0]) < 1e-6))

coef_l1 = np.array(coef_l1)  # shape (n_C, n_features)
coef_l2 = np.array(coef_l2)

print("L1 Sparsity — Number of Zero Coefficients at Each C:")
print(f"{'C':<8} {'λ=1/C':<10} {'Zeros (L1)':<14} {'Zeros (L2)'}")
print("-" * 45)
for i, C in enumerate(C_vals):
    zeros_l2 = np.sum(np.abs(coef_l2[i]) < 1e-6)
    print(f"{C:<8.2f} {1/C:<10.3f} {zero_count_l1[i]:<14} {zeros_l2}")

print(f"\nL1 can zero out coefficients exactly; L2 only approaches zero asymptotically.")

# Coefficient path plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('L1 vs L2 Regularization Paths\n'
             'L1 drives coefficients to exactly zero; L2 only shrinks them',
             fontsize=13, fontweight='bold')

lambda_axis = [1/C for C in C_vals]  # convert to λ for x-axis
feat_colors = [SPAM_COLOR, '#3498db', '#2ecc71', '#9b59b6']

# L1 path
ax = axes[0]
for i, name in enumerate(feat_names):
    ax.plot(lambda_axis, coef_l1[:, i], marker='o', lw=2,
            color=feat_colors[i], label=name)
ax.axhline(0, color='black', lw=1.2, linestyle='--')
ax.set_xscale('log')
ax.set_xlabel('λ = 1/C (regularization strength)')
ax.set_ylabel('Coefficient Value')
ax.set_title('L1 Regularization Path\nCoefficients hit EXACTLY zero (sparsity!)')
ax.legend(fontsize=9)
ax.fill_between([lambda_axis[-1], lambda_axis[0]], -0.05, 0.05,
               alpha=0.15, color='gray', label='Zero band')

# L2 path
ax2 = axes[1]
for i, name in enumerate(feat_names):
    ax2.plot(lambda_axis, coef_l2[:, i], marker='s', lw=2,
             color=feat_colors[i], label=name, linestyle='--')
ax2.axhline(0, color='black', lw=1.2, linestyle='--')
ax2.set_xscale('log')
ax2.set_xlabel('λ = 1/C (regularization strength)')
ax2.set_ylabel('Coefficient Value')
ax2.set_title('L2 Regularization Path\nCoefficients approach zero but never reach it')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 7: Learning Rate Effects — Slow, Good, and Diverging

In [ ]:
# ============================================================
# Cell 9: Learning Rate Comparison
# α = 0.001 (too slow), 0.1 (good), 2.0 (diverges)
# Plot 3 loss curves to show the dramatic difference
# ============================================================

learning_rates = {
    '0.001 (too slow)':  0.001,
    '0.1   (good)':      0.1,
    '2.0   (diverges)':  2.0,
}
lr_colors = [C_SLOW, C_GOOD, C_FAST]
lr_styles = [':', '-', '--']

results = {}
for (label, lr), color, style in zip(learning_rates.items(), lr_colors, lr_styles):
    model = LogisticRegressionGD(lr=lr, n_iter=500, lambda_=0.0)
    model.fit(X_train, y_train)
    results[label] = {
        'losses': model.loss_history_,
        'color':  color,
        'style':  style,
        'acc':    accuracy_score(y_test, model.predict(X_test)),
        'final_loss': model.loss_history_[-1],
    }

print("Learning Rate Comparison (500 iterations each):")
print(f"{'Learning Rate':<28} {'Final Loss':>12} {'Test Acc':>10}")
print("-" * 52)
for label, res in results.items():
    final_loss = res['final_loss']
    note = '← diverged!' if final_loss > 10 else ('← slow convergence' if '0.001' in label else '← optimal')
    print(f"{label:<28} {final_loss:>12.4f} {res['acc']*100:>9.1f}%  {note}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Learning Rate Effects on Gradient Descent\n'
             'Too small = slow; too large = diverges; just right = converges fast',
             fontsize=13, fontweight='bold')

# Full 500 iterations
ax = axes[0]
for label, res in results.items():
    losses = np.minimum(res['losses'], 5.0)  # clip diverged losses for readability
    ax.plot(losses, color=res['color'], lw=2, linestyle=res['style'],
            label=f'α={label.split("(")[0].strip()}')
ax.set_xlabel('Iteration')
ax.set_ylabel('CE Loss (clipped at 5)')
ax.set_title('All 500 Iterations')
ax.legend()

# Zoom: first 100 iterations
ax2 = axes[1]
for label, res in results.items():
    losses = np.minimum(res['losses'][:100], 5.0)
    ax2.plot(losses, color=res['color'], lw=2, linestyle=res['style'],
             label=f'α={label.split("(")[0].strip()}')
ax2.set_xlabel('Iteration (first 100)')
ax2.set_ylabel('CE Loss (clipped at 5)')
ax2.set_title('First 100 Iterations (zoom)')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10: Gradient update animation — visualize parameter trajectory
# Shows where gradient descent goes on the 2D loss surface
# for 3 different learning rates
# ============================================================

# Use 1-feature model for clear 1D trajectory visualization
X_1f  = X_train[:, 1:2]   # exclamation_marks only
X_1fb = np.column_stack([np.ones(len(X_1f)), X_1f])

def bce_1d(theta1, theta0=0.0):
    """BCE loss for 1-feature model as function of theta1."""
    z     = X_1fb @ np.array([theta0, theta1])
    y_hat = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    y_s   = np.clip(y_hat, 1e-15, 1-1e-15)
    return -np.mean(y_train * np.log(y_s) + (1-y_train) * np.log(1-y_s))

# Loss surface as function of theta1 (1D)
t1_range = np.linspace(-3, 5, 300)
losses_1d = [bce_1d(t1) for t1 in t1_range]

# Run gradient descent from same starting point with 3 learning rates
def gd_1d_trajectory(lr, n_iter=40, t1_init=-2.5):
    """Run GD on 1-feature model, return (theta1 trajectory, loss trajectory)."""
    t0, t1 = 0.0, t1_init
    traj_t, traj_l = [t1], [bce_1d(t1)]
    for _ in range(n_iter):
        theta = np.array([t0, t1])
        z     = X_1fb @ theta
        y_hat = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        grad  = (1/len(y_train)) * X_1fb.T @ (y_hat - y_train)
        t0   -= lr * grad[0]
        t1   -= lr * grad[1]
        traj_t.append(t1)
        traj_l.append(bce_1d(t1))
    return traj_t, traj_l

traj_slow_t, traj_slow_l = gd_1d_trajectory(lr=0.05)
traj_good_t, traj_good_l = gd_1d_trajectory(lr=0.5)
traj_fast_t, traj_fast_l = gd_1d_trajectory(lr=2.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gradient Descent Trajectories on the CE Loss Surface\n'
             '(1-feature model: exclamation_marks weight)',
             fontsize=13, fontweight='bold')

# Panel 1: Loss surface + trajectory points
ax = axes[0]
ax.plot(t1_range, losses_1d, color='gray', lw=2, label='Loss surface')
for (traj_t, traj_l, label, color, style) in [
    (traj_slow_t, traj_slow_l, 'α=0.05 (slow)', C_SLOW,  ':'),
    (traj_good_t, traj_good_l, 'α=0.5  (good)', C_GOOD,  '-'),
    (traj_fast_t, traj_fast_l, 'α=2.5  (oscillates)', C_FAST, '--'),
]:
    ax.plot(traj_t[:20], traj_l[:20], marker='o', markersize=4,
            color=color, lw=2, linestyle=style, label=label, alpha=0.85)
ax.set_xlabel('θ₁ (exclamation weight)')
ax.set_ylabel('CE Loss')
ax.set_title('Trajectory on Loss Surface (first 20 steps)')
ax.set_ylim(min(losses_1d) - 0.05, max(losses_1d[:50]) + 0.05)
ax.legend(fontsize=9)

# Panel 2: Loss vs iteration for all 3
ax2 = axes[1]
ax2.plot(traj_slow_l, color=C_SLOW, lw=2, linestyle=':', label='α=0.05 (slow)')
ax2.plot(traj_good_l, color=C_GOOD, lw=2, linestyle='-',  label='α=0.5  (good)')
ax2.plot(np.minimum(traj_fast_l, max(traj_good_l)*2),
         color=C_FAST, lw=2, linestyle='--', label='α=2.5  (oscillates)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('CE Loss')
ax2.set_title('Loss vs Iteration\n(fast LR loss clipped for readability)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 8: Decision Boundary Visualization

In [ ]:
# ============================================================
# Cell 11: Decision Boundary Plot
# Show how the learned coefficients define the boundary between
# spam and ham regions in 2D feature space
# ============================================================

# Train 4 models with different regularization strengths
reg_configs = [
    (0.0,   'No regularization (λ=0)'),
    (0.1,   'Light L2 (λ=0.1)'),
    (1.0,   'Moderate L2 (λ=1.0)'),
    (10.0,  'Strong L2 (λ=10.0)'),
]

# Use 2 features for 2D visualization: word_count and exclamation_marks
X_2f = X_train[:, :2]
X_2f_test = X_test[:, :2]

# Build the decision boundary mesh
x_min, x_max = X_2f[:, 0].min()-0.5, X_2f[:, 0].max()+0.5
y_min, y_max = X_2f[:, 1].min()-0.5, X_2f[:, 1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150),
                      np.linspace(y_min, y_max, 150))
grid = np.column_stack([xx.ravel(), yy.ravel()])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat  = axes.ravel()
fig.suptitle('Decision Boundaries: Effect of L2 Regularization\n'
             'Features: word_count (x) vs exclamation_marks (y)',
             fontsize=13, fontweight='bold')

for ax, (lam, title) in zip(axes_flat, reg_configs):
    # Train model with this regularization
    m = LogisticRegressionGD(lr=0.15, n_iter=1200, lambda_=lam)
    m.fit(X_2f, y_train)

    # Predict on grid to draw decision regions
    Z = m.predict_proba(grid).reshape(xx.shape)

    # Plot probability regions
    ax.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.5, vmin=0, vmax=1)
    ax.contour( xx, yy, Z, levels=[0.5], colors='black', linewidths=2.0,
                linestyles='-')  # decision boundary at p=0.5

    # Scatter training data
    ax.scatter(X_2f[y_train==1, 0], X_2f[y_train==1, 1],
               c=SPAM_COLOR, s=20, alpha=0.5, edgecolors='none', label='Spam (train)')
    ax.scatter(X_2f[y_train==0, 0], X_2f[y_train==0, 1],
               c=HAM_COLOR,  s=20, alpha=0.5, edgecolors='none', label='Ham (train)')

    acc_2f = accuracy_score(y_test, m.predict(X_2f_test))
    theta_str = f"θ=({m.intercept_:.2f}, {m.coef_[0]:.2f}, {m.coef_[1]:.2f})"
    ax.set_title(f"{title}\n{theta_str} | Test acc: {acc_2f:.1%}")
    ax.set_xlabel('word_count (scaled)')
    ax.set_ylabel('exclamation_marks (scaled)')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    if ax == axes_flat[0]:
        ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()
print("Black line = decision boundary (P(spam)=0.5).")
print("Green regions = model predicts ham; red regions = model predicts spam.")
print("Stronger regularization → boundary orientation changes but remains linear.")

In [ ]:
# ============================================================
# Cell 12: Gradient Derivation Visualization
# Show the three chain-rule components and the final elegant formula
# ============================================================

# Illustrate the three components of the gradient for a single sample
theta_demo = np.array([0.1, 0.5, -0.3, 0.8, 0.2])  # bias + 4 weights
X_b_demo   = np.column_stack([np.ones(len(X_train)), X_train])

z_demo     = X_b_demo @ theta_demo
sigma_z    = 1 / (1 + np.exp(-z_demo))          # σ(z) = predictions
sigma_prime = sigma_z * (1 - sigma_z)            # σ'(z) = sigmoid derivative
error       = sigma_z - y_train                  # ŷ - y

# Gradient via chain rule:
#   ∂L/∂θ = (1/n) * Xᵀ * (ŷ-y)
# Let's break it down step by step

print("Chain Rule Gradient Decomposition (sample statistics)")
print("=" * 60)
print(f"\nStep 1: ∂L/∂ŷ = -(1/n)(y/ŷ - (1-y)/(1-ŷ))")
dL_dhat = -(1/len(y_train)) * (y_train / (sigma_z+1e-15) - (1-y_train) / (1-sigma_z+1e-15))
print(f"  Mean |∂L/∂ŷ|: {np.mean(np.abs(dL_dhat)):.4f}")

print(f"\nStep 2: ∂ŷ/∂z = σ(z)(1-σ(z))")
print(f"  Mean σ'(z): {sigma_prime.mean():.4f}  (ranges [0, 0.25])")

print(f"\nStep 3: Chain rule product ∂L/∂ŷ · ∂ŷ/∂z")
product_1_2 = dL_dhat * sigma_prime
print(f"  Simplifies to: (1/n)(ŷ - y)  ← σ(z)(1-σ(z)) cancels!")
print(f"  Mean |ŷ - y| (error): {np.mean(np.abs(error)):.4f}")

print(f"\nFinal gradient: (1/n) Xᵀ(ŷ-y)")
final_grad = (1/len(y_train)) * X_b_demo.T @ error
print(f"  Gradient shape: {final_grad.shape}")
feat_labels = ['bias', 'word_count', 'exclamation', 'capital_ratio', 'link_count']
for name, g in zip(feat_labels, final_grad):
    direction = 'increase θ' if g < 0 else 'decrease θ'
    print(f"  ∂L/∂{name:<16}: {g:>8.5f}  → {direction}'s prediction toward truth'")

print(f"\nKey insight: gradient = 0 when all (ŷᵢ - yᵢ) = 0 (perfect predictions)")
print(f"The model's parameters stop updating exactly when it has learned perfectly.")

In [ ]:
# ============================================================
# Cell 13: Full Notebook Summary
# ============================================================

# Final model: use best lambda based on regularization path
best_lam_idx = np.argmax([accuracy_score(y_test, 
                           LogisticRegressionGD(lr=0.1, n_iter=1500, lambda_=l).fit(X_train, y_train).predict(X_test))
                          for l in lambda_vals])
best_lam     = lambda_vals[best_lam_idx]

final_model = LogisticRegressionGD(lr=0.1, n_iter=1500, lambda_=best_lam)
final_model.fit(X_train, y_train)
final_acc   = accuracy_score(y_test, final_model.predict(X_test))
final_ce    = log_loss(y_test, np.clip(final_model.predict_proba(X_test), 1e-15, 1-1e-15))

print("=" * 65)
print("NOTEBOOK 4 SUMMARY — Gradient Descent on Cross-Entropy")
print("=" * 65)

print("\n1. GRADIENT FORMULA (the beautiful result)")
print("   ∂L/∂θ = (1/n) Xᵀ(ŷ - y)")
print("   Plain English: average prediction error weighted by features")
print("   Identical form to linear regression gradient!")

print("\n2. UPDATE RULE")
print("   θ ← θ - α · (1/n) Xᵀ(σ(Xθ) - y)")
print("   With L2: θ ← θ - α · [(1/n) Xᵀ(ŷ-y) + λθ]  (not applied to bias)")

print("\n3. GRADIENT CHECK")
print(f"   Max diff (analytical vs numerical): {max_diff:.2e} (threshold: 1e-4)")
print(f"   Result: {'PASSED ✓' if max_diff < 1e-4 else 'FAILED ✗'}")

print("\n4. vs sklearn")
print(f"   np.allclose(our coefs, sklearn coefs, atol=0.05): "
      f"{np.allclose(model_scratch.coef_, clf_sklearn.coef_[0], atol=0.05)}")
print(f"   (Small diffs expected: GD vs L-BFGS optimizer)")

print("\n5. REGULARIZATION")
print(f"   Best λ from path: {best_lam}  (sklearn C = {1/best_lam if best_lam > 0 else 'inf'})")
print(f"   Final test accuracy: {final_acc*100:.1f}%")
print(f"   Final test CE loss:  {final_ce:.4f}")
print(f"   L1 can zero-out irrelevant features; L2 only shrinks them")

print("\n6. LEARNING RATE LESSONS")
print("   α = 0.001: converges but needs many iterations (wasteful)")
print("   α = 0.1:   good balance of speed and stability")
print("   α = 2.0:   diverges — loss oscillates and blows up")
print("   Rule of thumb: try α ∈ {0.001, 0.01, 0.1, 0.3, 1.0} and monitor loss")

print("\n" + "=" * 65)

## Self-Check: Test Your Understanding

Attempt each question before expanding the answer.

---

**Question 1:** The gradient is $\nabla_\theta L = \frac{1}{n} X^T(\hat{y} - y)$. When the model is confidently correct (ŷ ≈ y for most samples), what happens to the gradient? Why is this the right behavior?

<details>
<summary>Click to reveal Answer 1</summary>

**Answer 1:**

When the model is confidently correct, the prediction errors $(\hat{y}_i - y_i)$ are **close to zero** for most samples:
- Spam emails where ŷ ≈ 1: error ≈ 1 - 1 = 0
- Ham emails where ŷ ≈ 0: error ≈ 0 - 0 = 0

Therefore $X^T(\hat{y} - y) \approx X^T \mathbf{0} \approx \mathbf{0}$, and the gradient approaches **zero**.

**Why this is correct behavior:** A zero gradient means the parameter update $\theta \leftarrow \theta - \alpha \cdot \mathbf{0}$ leaves $\theta$ unchanged — the model has converged. This makes intuitive sense: if the model is already predicting correctly with high confidence, there is nothing to improve and the optimization should stop.

This is one of the most elegant properties of the cross-entropy gradient: it naturally **self-terminates** when the model has learned the data well, rather than requiring an explicit stopping criterion.

**Contrast:** A model that is confidently wrong (ŷ = 1 when y = 0) would produce errors of ±1 and thus a large gradient — causing large parameter updates to correct the mistake. The gradient is both the learning signal and the termination criterion.
</details>

---

**Question 2:** sklearn uses `C` (inverse regularization strength), not `λ`. If you increase `C` from 1.0 to 100.0, does this give stronger or weaker regularization? What happens to the learned coefficients?

<details>
<summary>Click to reveal Answer 2</summary>

**Answer 2:**

**Increasing C = Weaker regularization.**

The relationship is: $C = \frac{1}{\lambda}$, so:
- $C = 1.0 \to \lambda = 1.0$ (moderate regularization)
- $C = 100.0 \to \lambda = 0.01$ (very weak regularization)

**What happens to coefficients:** As C increases (λ decreases), the regularization penalty $\frac{\lambda}{2}\|\theta\|^2$ shrinks. The model is less penalized for large weights, so coefficients can become **larger in magnitude** to fit the training data more precisely.

**Practical memory trick:** Think "C for Complexity" — larger C allows more model complexity (bigger weights, tighter fit to training data, higher risk of overfitting). Smaller C forces simpler models with smaller weights.

| C value | λ = 1/C | Regularization | Coefficient size | Overfitting risk |
|---------|---------|----------------|------------------|------------------|
| 0.01    | 100.0   | Very strong    | Very small       | Very low         |
| 1.0     | 1.0     | Moderate       | Moderate         | Moderate         |
| 100.0   | 0.01    | Very weak      | Large            | High             |
| 1e6     | ≈0      | None           | Unrestricted     | Very high        |
</details>

---

**Question 3:** Your spam classifier needs to work with 10,000 word features from a bag-of-words representation. Should you use L1 or L2 regularization, and why? What would happen if you used the other one?

<details>
<summary>Click to reveal Answer 3</summary>

**Answer 3:**

**Prefer L1 regularization for this use case.** Here is the reasoning:

**Why L1 is better for 10,000 word features:**

1. **Sparsity:** Most words are not predictive of spam. Common words like "the", "and", "is" appear equally in spam and ham. L1 drives non-predictive word weights to **exactly zero**, effectively selecting only the truly discriminative words ("free", "winner", "click", "urgent", etc.).

2. **Interpretability:** After L1 training, you can inspect the non-zero weights to understand exactly which words the model is using — valuable for debugging and trust.

3. **Efficiency:** Only storing the ~50–200 non-zero coefficients instead of all 10,000 saves memory and speeds up predictions.

4. **Automatic feature selection:** L1 eliminates the need for a separate feature selection step.

**What L2 would do instead:**
- All 10,000 coefficients would remain non-zero (just very small)
- The model would use every word slightly, even noise words
- Results can still be good, but the model is less interpretable and less efficient
- L2 tends to perform better when many features are genuinely informative (e.g., continuous sensor readings), while L1 excels when most features are irrelevant

**In practice:** Many production spam filters use **Elastic Net** (a weighted combination of L1 and L2) to get the sparsity benefits of L1 while avoiding L1's instability when features are correlated (e.g., "free offer" and "click free" are highly correlated word pairs).
</details>